In [ ]:
import torch
import joblib
from transformers import AutoTokenizer, AutoModelForQuestionAnswering

In [ ]:
qa_model_name = "deepset/bert-base-cased-squad2"

tokenizer = AutoTokenizer.from_pretrained(qa_model_name)
qa_model = AutoModelForQuestionAnswering.from_pretrained(qa_model_name)

qa_model.eval()

In [ ]:
rf_model = joblib.load("../results/reliability_classifier.pkl")

print("Reliability classifier loaded")

In [ ]:
def demo_question(context, question):

    inputs = tokenizer(question, context, return_tensors="pt")

    with torch.no_grad():
        outputs = model(**inputs)

    start_logits = outputs.start_logits
    end_logits = outputs.end_logits

    start_index = torch.argmax(start_logits)
    end_index = torch.argmax(end_logits)

    tokens = tokenizer.convert_ids_to_tokens(inputs["input_ids"][0])
    answer = tokenizer.convert_tokens_to_string(tokens[start_index:end_index+1])

    # ---- compute signals ----
    start_probs = torch.softmax(start_logits, dim=-1)
    end_probs = torch.softmax(end_logits, dim=-1)

    span_prob = (start_probs[0][start_index] * end_probs[0][end_index]).item()

    start_entropy = -(start_probs * torch.log(start_probs + 1e-12)).sum().item()
    end_entropy = -(end_probs * torch.log(end_probs + 1e-12)).sum().item()

    top2 = torch.topk(start_probs[0], 2).values
    confidence_margin = (top2[0] - top2[1]).item()

    entropy_mean = (start_entropy + end_entropy) / 2
    entropy_diff = abs(start_entropy - end_entropy)

    features = [[
        span_prob,
        start_entropy,
        end_entropy,
        confidence_margin,
        entropy_mean,
        entropy_diff
    ]]

    score = rf_model.predict_proba(features)[0][1]

    print("Question:", question)
    print("Answer:", answer)
    print("Reliability Score:", round(score,3))

In [ ]:
context = "Tesla was founded by Elon Musk in 2003."

question = "Who founded Tesla?"

demo_question(context, question)

In [ ]:
context = "Tesla was founded by Elon Musk in 2003."

question = "Who invented electricity in Tesla company?"

demo_question(context, question)

## Upgraded model

In [2]:
import torch
import joblib
from transformers import AutoTokenizer, AutoModelForQuestionAnswering

In [3]:
qa_model_name = "deepset/bert-base-cased-squad2"

tokenizer = AutoTokenizer.from_pretrained(qa_model_name)
qa_model = AutoModelForQuestionAnswering.from_pretrained(qa_model_name)

qa_model.eval()

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForQuestionAnswering LOAD REPORT from: deepset/bert-base-cased-squad2
Key                      | Status     |  | 
-------------------------+------------+--+-
bert.pooler.dense.weight | UNEXPECTED |  | 
bert.pooler.dense.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


BertForQuestionAnswering(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(28996, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e-12, elem

In [4]:
rf_model = joblib.load("../results/reliability_classifier.pkl")

In [5]:
def demo_question(context, question):

    inputs = tokenizer(question, context, return_tensors="pt")

    with torch.no_grad():
        outputs = qa_model(**inputs)

    start_logits = outputs.start_logits
    end_logits = outputs.end_logits

    start_index = torch.argmax(start_logits)
    end_index = torch.argmax(end_logits)

    tokens = tokenizer.convert_ids_to_tokens(inputs["input_ids"][0])
    answer = tokenizer.convert_tokens_to_string(tokens[start_index:end_index+1])

    # ---- Handle NO ANSWER case ----
    if answer.strip() == "[CLS]":
        print("\n---------------------------")
        print("Question:", question)
        print("Answer: No answer found")
        print("Reliability: LOW")
        return

    # ---- Compute signals ----
    start_probs = torch.softmax(start_logits, dim=-1)
    end_probs = torch.softmax(end_logits, dim=-1)

    span_prob = (start_probs[0][start_index] * end_probs[0][end_index]).item()

    # normalized entropy
    start_entropy = -(start_probs * torch.log(start_probs + 1e-12)).sum().item() / start_probs.shape[-1]
    end_entropy = -(end_probs * torch.log(end_probs + 1e-12)).sum().item() / end_probs.shape[-1]

    top2 = torch.topk(start_probs[0], 2).values
    confidence_margin = (top2[0] - top2[1]).item()

    entropy_mean = (start_entropy + end_entropy) / 2
    entropy_diff = abs(start_entropy - end_entropy)

    features = [[
        span_prob,
        start_entropy,
        end_entropy,
        confidence_margin,
        entropy_mean,
        entropy_diff
    ]]

    score = 1 - rf_model.predict_proba(features)[0][1]

    # ---- Convert score to label ----
    if score > 0.6:
        label = "High"
    elif score > 0.4:
        label = "Medium"
    else:
        label = "Low"

    # ---- Final Output ----
    print("\n---------------------------")
    print("Question:", question)
    print("Answer:", answer)
    print("Reliability Score:", round(score, 3))
    print("Reliability Level:", label)

In [6]:
context = """
Tesla, Inc. is an American electric vehicle company founded in 2003.
The company was founded by engineers Martin Eberhard and Marc Tarpenning.
Elon Musk later joined as an investor and became the public face of the company.
Tesla designs electric vehicles and energy solutions.
"""

In [7]:
demo_question(context, "Who became the public face of Tesla?")


---------------------------
Question: Who became the public face of Tesla?
Answer: Elon Musk
Reliability Score: 0.897
Reliability Level: High


/opt/anaconda3/envs/qa_project/lib/python3.10/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(


In [9]:
demo_question(context, "Who invented electricity in Tesla?")


---------------------------
Question: Who invented electricity in Tesla?
Answer: No answer found
Reliability: LOW
